In [1]:
import pandas as pd
import numpy as np
import pickle
from sklearn.inspection import partial_dependence

# 1. Load the winning model pipeline and training data
with open('../deployment/model_artifacts_002.pkl', 'rb') as f:
    artifacts = pickle.load(f)

best_model = artifacts['model']
train_df = pd.read_csv('../data/train_data.csv')
X_train = train_df.drop(columns='Sold_Price')

In [2]:
# 2. Select the continuous variables you want to visualize
# (Categoricals/Target Encoded features are less interpretable on a PDP line chart)
features_to_plot = [
    'Mileage', 'car_age', 'Engine_Displacement_L', 'mileage_per_year', "Gears"
]

pdp_results = []

# Fill NaNs temporarily so scikit-learn can calculate the percentiles
X_train_pdp = X_train.copy()
X_train_pdp['mileage_per_year'] = X_train_pdp['mileage_per_year'].fillna(X_train_pdp['mileage_per_year'].median())

print("Calculating Partial Dependency Plots...")
for feature in features_to_plot:
    # Use the filled dataset for the PDP generator
    pdp = partial_dependence(best_model, X_train_pdp, features=[feature], grid_resolution=50)
    
    # Accommodate scikit-learn version differences for the grid key
    grid_key = 'values' if 'values' in pdp else 'grid_values'
    grid_values = pdp[grid_key][0]
    
    # The model targets log-space, so we inverse transform to real dollars
    avg_predictions_dollars = np.expm1(pdp['average'][0])
    
    for val, pred in zip(grid_values, avg_predictions_dollars):
        pdp_results.append({
            'Feature': feature,
            'Feature_Value': val,
            'Predicted_Price': pred
        })

Calculating Partial Dependency Plots...


In [3]:
# 3. Export to the frontend data directory
pdp_df = pd.DataFrame(pdp_results)
pdp_df.to_csv('../data/frontend_data/pdp_data.csv', index=False)
print("Saved PDP data to ../data/frontend_data/pdp_data.csv")

Saved PDP data to ../data/frontend_data/pdp_data.csv
